# CVS metadata-clean eval (C1/C2/C3) — grouped by schema version

This notebook loads prediction JSONLs produced by `metadata_clean_pipeline_param_multi.py` for **C1/C2/C3** and reports metrics by:

- `criterion`: c1/c2/c3  
- `condition`: with_metadata / no_metadata / cues_only  
- `schema_version`: v1/v2/v3/... inferred from filename (if no `_vN` suffix, treated as **v1**)

It also loads GT from `data/CVS_Challenge_SAGES_v1/<split>/labels/<video_id>/frame.csv` and computes:
- Accuracy
- Confusion counts (tp/fp/tn/fn)
- AP (average precision) using binary label or continuous `score`


In [1]:

from __future__ import annotations

import csv
import json
import math
import random
import statistics
import re
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Literal

import pandas as pd
from sklearn.metrics import average_precision_score

# -----------------------------
# CONFIG: edit this cell only
# -----------------------------

# REPO_ROOT = Path.cwd().resolve()
REPO_ROOT = Path("/shared_data0/weiqiuy/lvlm_cvs_reasoning").resolve()

MANIFEST_NAME: Optional[str] = "train_dev5"  # set to None to disable filename filtering
MANIFEST_JSONL = (REPO_ROOT / "data" / f"{MANIFEST_NAME}.jsonl").resolve()

GT_LABELS_ROOT = (REPO_ROOT / "data" / "CVS_Challenge_SAGES_v1").resolve()

VOTE_RULE: Literal["any", "majority"] = "majority"   # majority over 3 raters => >=2

PRED_ROOTS = {
    "c1": (REPO_ROOT / "outputs" / "c1_pred").resolve(),
    "c2": (REPO_ROOT / "outputs" / "c2_pred").resolve(),
    "c3": (REPO_ROOT / "outputs" / "c3_pred").resolve(),
}

# Conditions and filename patterns inside each criterion folder.
CONDITIONS = {
    "with_metadata": "*__with_meta__*.jsonl",
    "no_metadata": "*__no_meta__*.jsonl",
    "cues_only": "*__cues_only__*.jsonl",
}

print("REPO_ROOT:", REPO_ROOT)
print("MANIFEST_JSONL:", MANIFEST_JSONL)


REPO_ROOT: /shared_data0/weiqiuy/lvlm_cvs_reasoning
MANIFEST_JSONL: /shared_data0/weiqiuy/lvlm_cvs_reasoning/data/train_dev5.jsonl


In [4]:

# -----------------------------
# Helpers
# -----------------------------
def iter_jsonl(path: Path) -> Iterable[Dict[str, Any]]:
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                yield json.loads(line)

def kf(video_id: str, split: str, frame_id: int) -> str:
    return f"{video_id}::{split}::{frame_id}"

def load_frame_gt(video_id: str, split: str, *, criterion: str) -> Dict[int, int]:
    p = (GT_LABELS_ROOT / split / "labels" / video_id / "frame.csv").resolve()
    if not p.exists():
        return {}
    out: Dict[int, int] = {}
    with p.open("r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            fid = int(row["frame_id"])
            votes = [
                int(row.get(f"{criterion}_rater1", 0) or 0),
                int(row.get(f"{criterion}_rater2", 0) or 0),
                int(row.get(f"{criterion}_rater3", 0) or 0),
            ]
            if VOTE_RULE == "any":
                out[fid] = 1 if any(v == 1 for v in votes) else 0
            else:
                out[fid] = 1 if sum(votes) >= 2 else 0
    return out

def build_gt_index(*, criterion: str) -> Dict[str, int]:
    gt_idx: Dict[str, int] = {}
    manifest_rows = list(iter_jsonl(MANIFEST_JSONL))
    for rec in manifest_rows:
        vid = rec.get("id") or rec.get("video_id")
        sp = rec.get("split", "train")
        if vid is None:
            continue
        per = load_frame_gt(str(vid), str(sp), criterion=criterion)
        for fid, y in per.items():
            gt_idx[kf(str(vid), str(sp), int(fid))] = int(y)
    return gt_idx

def _safe_float(x: Any, default: float) -> float:
    try:
        v = float(x)
        if math.isnan(v) or math.isinf(v):
            return default
        return v
    except Exception:
        return default

def confusion(y_true: List[int], y_pred: List[int]) -> Dict[str, int]:
    tp = sum(1 for yt, yp in zip(y_true, y_pred) if yt == 1 and yp == 1)
    fp = sum(1 for yt, yp in zip(y_true, y_pred) if yt == 0 and yp == 1)
    tn = sum(1 for yt, yp in zip(y_true, y_pred) if yt == 0 and yp == 0)
    fn = sum(1 for yt, yp in zip(y_true, y_pred) if yt == 1 and yp == 0)
    return {"tp": tp, "fp": fp, "tn": tn, "fn": fn}



def bootstrap_std(
    *,
    y_true: List[int],
    y_pred: Optional[List[int]] = None,
    y_score: Optional[List[float]] = None,
    metric: Literal["accuracy", "ap"],
    n_boot: int = 4,
) -> float:
    if not y_true:
        return float("nan")
    n = len(y_true)
    vals: List[float] = []
    for _ in range(n_boot):
        idx = [random.randrange(n) for _ in range(n)]
        if metric == "accuracy":
            if y_pred is None:
                return float("nan")
            yt = [y_true[i] for i in idx]
            yp = [y_pred[i] for i in idx]
            acc = sum(1 for a, b in zip(yt, yp) if a == b) / len(yt)
            vals.append(acc)
        else:
            if y_score is None:
                return float("nan")
            yt = [y_true[i] for i in idx]
            ys = [y_score[i] for i in idx]
            if len(set(yt)) <= 1:
                continue
            vals.append(average_precision_score(yt, ys))
    if not vals:
        return float("nan")
    return float(statistics.pstdev(vals))

def eval_rows(
    rows: List[Dict[str, Any]],
    *,
    criterion: str,
    gt_idx: Dict[str, int],
    score_mode: Literal["binary", "continuous"],
    drop_parse_fail: bool = True,
) -> Dict[str, Any]:
    y_true: List[int] = []
    y_pred: List[int] = []
    y_score: List[float] = []

    n_total = 0
    n_missing_gt = 0
    n_parse_fail = 0

    for r in rows:
        n_total += 1
        parse_ok = bool(r.get("parse_ok", False))
        if drop_parse_fail and not parse_ok:
            n_parse_fail += 1
            continue

        vid = r.get("video_id")
        sp = r.get("split")
        fid = r.get("frame_id")
        if vid is None or sp is None or fid is None:
            continue

        kk = kf(str(vid), str(sp), int(fid))
        if kk not in gt_idx:
            n_missing_gt += 1
            continue

        pred = r.get("pred") or {}
        yt = int(gt_idx[kk])

        yp = int(_safe_float(pred.get(criterion, 0), 0.0))
        yp = 1 if yp == 1 else 0

        if score_mode == "binary":
            sc = float(yp)
        else:
            sc = _safe_float(pred.get("score", 0.5), 0.5)
            sc = max(0.0, min(1.0, sc))

        y_true.append(yt)
        y_pred.append(yp)
        y_score.append(sc)

    acc = sum(1 for yt, yp in zip(y_true, y_pred) if yt == yp) / len(y_true) if y_true else 0.0
    conf = confusion(y_true, y_pred)
    ap_global = average_precision_score(y_true, y_score) if len(set(y_true)) > 1 else float("nan")

    acc_std = bootstrap_std(y_true=y_true, y_pred=y_pred, metric="accuracy")
    ap_std = bootstrap_std(y_true=y_true, y_score=y_score, metric="ap")

    return {
        "n_total_rows": n_total,
        "n_used": len(y_true),
        "n_parse_fail_dropped": n_parse_fail if drop_parse_fail else 0,
        "n_missing_gt": n_missing_gt,
        "accuracy": acc,
        "accuracy_std": acc_std,
        **conf,
        "AP_global": ap_global,
        "AP_global_std": ap_std,
    }

_VERSION_RE = re.compile(r"(?:^|_)v(\d+)\.jsonl$", re.IGNORECASE)
_MODEL_SPLIT_RE = re.compile(r"_v\d+\.jsonl$", re.IGNORECASE)
_NO_MODEL_TOKENS = {"train", "dev", "train_dev", "val", "test"}

def infer_schema_version_from_filename(p: Path) -> str:
    m = _VERSION_RE.search(p.name)
    if m:
        return f"v{int(m.group(1))}"
    return "v1"

def infer_model_from_filename(p: Path, *, default_model: str = "gpt-4.1-mini") -> str:
    m = _MODEL_SPLIT_RE.search(p.name)
    if not m:
        return default_model
    base = p.name[: m.start()]
    if "__" not in base:
        return default_model
    model = base.split("__")[-1].strip()
    if not model or model in _NO_MODEL_TOKENS:
        return default_model
    return model


def _manifest_token(name: str) -> str:
    return f"_{name.lower()}_"

def manifest_filter_ok(p: Path, manifest_name: Optional[str]) -> bool:
    if not manifest_name:
        return True
    return _manifest_token(manifest_name) in p.name.lower()

def find_latest_matching_by_version_and_model(
    root: Path,
    pattern: str,
    *,
    default_model: str = "gpt-4.1-mini",
    manifest_name: Optional[str] = None,
) -> Dict[tuple[str, str], Path]:
    out: Dict[tuple[str, str], Path] = {}
    matches = [p for p in root.glob(pattern) if manifest_filter_ok(p, manifest_name)]
    if not matches:
        return out

    buckets: Dict[tuple[str, str], List[Path]] = {}
    for p in matches:
        v = infer_schema_version_from_filename(p)
        model = infer_model_from_filename(p, default_model=default_model)
        buckets.setdefault((v, model), []).append(p)

    for key, ps in buckets.items():
        ps.sort(key=lambda p: p.stat().st_mtime, reverse=True)
        out[key] = ps[0]
    return out


def eval_from_lists(
    y_true: List[int],
    y_pred: List[int],
    y_score: List[float],
) -> Dict[str, Any]:
    acc = sum(1 for yt, yp in zip(y_true, y_pred) if yt == yp) / len(y_true) if y_true else 0.0
    conf = confusion(y_true, y_pred)
    ap_global = average_precision_score(y_true, y_score) if len(set(y_true)) > 1 else float("nan")
    acc_std = bootstrap_std(y_true=y_true, y_pred=y_pred, metric="accuracy")
    ap_std = bootstrap_std(y_true=y_true, y_score=y_score, metric="ap")
    return {
        "n_total_rows": len(y_true),
        "n_used": len(y_true),
        "n_parse_fail_dropped": 0,
        "n_missing_gt": 0,
        "accuracy": acc,
        "accuracy_std": acc_std,
        **conf,
        "AP_global": ap_global,
        "AP_global_std": ap_std,
    }


In [5]:
import os

# -----------------------------
# Load & evaluate all criteria, grouped by version and model
# -----------------------------
rows_out: List[Dict[str, Any]] = []

for criterion, pred_root in PRED_ROOTS.items():
    if not pred_root.exists():
        print(f"[WARN] Missing pred root for {criterion}: {pred_root}")
        continue

    print(f"\n== {criterion.upper()} ==")
    gt_idx = build_gt_index(criterion=criterion)
    print("GT frames indexed:", len(gt_idx))


    # Baselines for this criterion (use GT labels only)
    gt_labels = list(gt_idx.values())
    if gt_labels:
        y_true = [int(v) for v in gt_labels]
        # constant 0
        y_pred = [0 for _ in y_true]
        y_score = [0.0 for _ in y_true]
        m_base = eval_from_lists(y_true, y_pred, y_score)
        rows_out.append({
            "criterion": criterion,
            "condition": "baseline",
            "schema_version": "baseline",
            "model": "constant0",
            "pred_file": "baseline_constant0",
            **{f"bin_{k}": v for k, v in m_base.items()},
            **{f"cont_{k}": v for k, v in m_base.items()},
        })

        # random 50/50
        rng = random.Random(7)
        y_pred = [rng.randint(0, 1) for _ in y_true]
        y_score = [float(v) for v in y_pred]
        m_base = eval_from_lists(y_true, y_pred, y_score)
        rows_out.append({
            "criterion": criterion,
            "condition": "baseline",
            "schema_version": "baseline",
            "model": "random50",
            "pred_file": "baseline_random50",
            **{f"bin_{k}": v for k, v in m_base.items()},
            **{f"cont_{k}": v for k, v in m_base.items()},
        })
    for condition, pattern in CONDITIONS.items():
        by_ver_model = find_latest_matching_by_version_and_model(
            pred_root,
            pattern,
            manifest_name=MANIFEST_NAME,
        )
        if not by_ver_model:
            print(f"[WARN] No files for {criterion} / {condition} in {pred_root} (pattern={pattern})")
            continue

        def _vnum(v: str) -> int:
            try:
                return int(v[1:]) if v.startswith("v") else 0
            except Exception:
                return 0

        for (schema_version, model), p in sorted(by_ver_model.items(), key=lambda kv: (_vnum(kv[0][0]), kv[0][1])):
            rows = list(iter_jsonl(p))
            m_bin = eval_rows(rows, criterion=criterion, gt_idx=gt_idx, score_mode="binary", drop_parse_fail=True)
            m_cont = eval_rows(rows, criterion=criterion, gt_idx=gt_idx, score_mode="continuous", drop_parse_fail=True)

            rows_out.append({
                "criterion": criterion,
                "condition": condition,
                "schema_version": schema_version,
                "model": model,
                "pred_file": os.path.basename(p),
                **{f"bin_{k}": v for k, v in m_bin.items()},
                **{f"cont_{k}": v for k, v in m_cont.items()},
            })

df = pd.DataFrame(rows_out)
if not df.empty:
    def _vnum(v: str) -> int:
        try:
            return int(v[1:]) if v.startswith("v") else 0
        except Exception:
            return 0
    df["schema_vnum"] = df["schema_version"].map(_vnum)
    df = df.sort_values(["criterion", "model", "schema_vnum", "condition"]).drop(columns=["schema_vnum"]).reset_index(drop=True)

df



== C1 ==
GT frames indexed: 90

== C2 ==
GT frames indexed: 90

== C3 ==
GT frames indexed: 90


,criterion,condition,schema_version,model,pred_file,bin_n_total_rows,bin_n_used,bin_n_parse_fail_dropped,bin_n_missing_gt,bin_accuracy,...,cont_n_parse_fail_dropped,cont_n_missing_gt,cont_accuracy,cont_accuracy_std,cont_tp,cont_fp,cont_tn,cont_fn,cont_AP_global,cont_AP_global_std
0,c1,cues_only,v1,claude-opus-4-5-20251101,c1__cues_only__train_dev5__claude-opus-4-5-202...,90,90,0,0,0.766667,...,0,0,0.766667,0.037268,0,0,69,21,0.448423,0.102696
1,c1,no_metadata,v1,claude-opus-4-5-20251101,c1__no_meta__train_dev5__claude-opus-4-5-20251...,90,90,0,0,0.766667,...,0,0,0.766667,0.016434,0,0,69,21,0.466023,0.144051
2,c1,with_metadata,v1,claude-opus-4-5-20251101,c1__with_meta__train_dev5__claude-opus-4-5-202...,90,90,0,0,0.766667,...,0,0,0.766667,0.030807,0,0,69,21,0.580363,0.027073
3,c1,cues_only,v3,claude-opus-4-5-20251101,c1__cues_only__train_dev5__claude-opus-4-5-202...,90,90,0,0,0.766667,...,0,0,0.766667,0.040445,0,0,69,21,0.571996,0.079571
4,c1,no_metadata,v3,claude-opus-4-5-20251101,c1__no_meta__train_dev5__claude-opus-4-5-20251...,90,90,0,0,0.766667,...,0,0,0.766667,0.025307,0,0,69,21,0.466023,0.035685
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
112,c3,with_metadata,v1,gpt-5.1,c3__with_meta__train_dev5__gpt-5.1_v1.jsonl,90,89,1,0,0.685393,...,1,0,0.685393,0.042415,13,21,48,7,0.440402,0.154576
113,c3,baseline,baseline,random50,baseline_random50,90,90,0,0,0.455556,...,0,0,0.455556,0.039577,7,36,34,13,0.201421,0.043792
114,c3,cues_only,v3,with_prevpreds,c3__cues_only__train_dev5__Qwen-Qwen2.5-VL-32B...,90,90,0,0,0.300000,...,0,0,0.300000,0.053504,12,55,15,8,0.253421,0.045906
115,c3,no_metadata,v3,with_prevpreds,c3__no_meta__train_dev5__Qwen-Qwen2.5-VL-32B-I...,90,90,0,0,0.322222,...,0,0,0.322222,0.054997,19,60,10,1,0.273304,0.045772


In [6]:
import pandas as pd

# define condition order (edit if you want different)
cond_order = ["with_metadata", "no_metadata", "cues_only"]

# make condition a categorical so sorting follows your preferred order
df2 = df.copy()
df2["condition"] = pd.Categorical(df2["condition"], categories=cond_order, ordered=True)

# extract version number for proper numeric sorting (v1 < v2 < v10)
df2["schema_vnum"] = (
    df2["schema_version"]
      .astype(str)
      .str.extract(r"v(\d+)", expand=False)
      .fillna("1")
      .astype(int)
)

# sort: criterion -> model -> schema_version -> condition
df2 = (
    df2.sort_values(["criterion", "model", "schema_vnum", "condition"])
       .drop(columns=["schema_vnum"])
       .reset_index(drop=True)
)

cols = [
    "criterion","model","schema_version","condition",
    "bin_n_used","bin_accuracy","bin_accuracy_std","bin_tp","bin_fp","bin_tn","bin_fn","bin_AP_global","bin_AP_global_std",
    "cont_AP_global","cont_AP_global_std",
    "pred_file",
]

import pandas as pd

pd.set_option("display.max_rows", None)

from IPython.display import display

display(df2[cols])

pd.reset_option("display.max_rows")


,criterion,model,schema_version,condition,bin_n_used,bin_accuracy,bin_accuracy_std,bin_tp,bin_fp,bin_tn,bin_fn,bin_AP_global,bin_AP_global_std,cont_AP_global,cont_AP_global_std,pred_file
0,c1,claude-opus-4-5-20251101,v1,with_metadata,90,0.766667,0.031793,0,0,69,21,0.233333,0.014434,0.580363,0.027073,c1__with_meta__train_dev5__claude-opus-4-5-202...
1,c1,claude-opus-4-5-20251101,v1,no_metadata,90,0.766667,0.064967,0,0,69,21,0.233333,0.016434,0.466023,0.144051,c1__no_meta__train_dev5__claude-opus-4-5-20251...
2,c1,claude-opus-4-5-20251101,v1,cues_only,90,0.766667,0.040350,0,0,69,21,0.233333,0.017568,0.448423,0.102696,c1__cues_only__train_dev5__claude-opus-4-5-202...
3,c1,claude-opus-4-5-20251101,v3,with_metadata,90,0.766667,0.031914,0,0,69,21,0.233333,0.030429,0.558390,0.086931,c1__with_meta__train_dev5__claude-opus-4-5-202...
4,c1,claude-opus-4-5-20251101,v3,no_metadata,90,0.766667,0.031793,0,0,69,21,0.233333,0.012108,0.466023,0.035685,c1__no_meta__train_dev5__claude-opus-4-5-20251...
5,c1,claude-opus-4-5-20251101,v3,cues_only,90,0.766667,0.047791,0,0,69,21,0.233333,0.012108,0.571996,0.079571,c1__cues_only__train_dev5__claude-opus-4-5-202...
6,c1,claude-opus-4-5-20251101,v4,with_metadata,90,0.766667,0.035136,0,0,69,21,0.233333,0.013608,0.409074,0.076291,c1__with_meta__train_dev5__claude-opus-4-5-202...
7,c1,claude-opus-4-5-20251101,v4,no_metadata,90,0.766667,0.033333,0,0,69,21,0.233333,0.025307,0.466023,0.097960,c1__no_meta__train_dev5__claude-opus-4-5-20251...
8,c1,claude-opus-4-5-20251101,v4,cues_only,90,0.766667,0.009213,0,0,69,21,0.233333,0.054716,0.233333,0.016434,c1__cues_only__train_dev5__claude-opus-4-5-202...
9,c1,claude-opus-4-5-20251101,v41,with_metadata,90,0.766667,0.016434,0,0,69,21,0.233333,0.019837,0.477878,0.086254,c1__with_meta__train_dev5__claude-opus-4-5-202...


In [ ]:
df_full = df[
    (df["bin_n_used"] == df["bin_n_total_rows"]) &
    (df["bin_n_total_rows"] > 0)
].reset_index(drop=True)



In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

criteria = ["c1", "c2", "c3"]
condition = "no_metadata"

dff = df2[df2["condition"] == condition].copy()

# ensure numeric
num_cols = [
    "bin_accuracy", "bin_accuracy_std",
    "cont_AP_global", "cont_AP_global_std",
]
for col in num_cols:
    dff[col] = pd.to_numeric(dff[col], errors="coerce")

for c in criteria:
    sub = (
        dff[dff["criterion"] == c]
        .groupby("model", as_index=False)
        .agg(
            bin_accuracy=("bin_accuracy", "mean"),
            bin_accuracy_std=("bin_accuracy_std", "mean"),
            cont_AP_global=("cont_AP_global", "mean"),
            cont_AP_global_std=("cont_AP_global_std", "mean"),
        )
    )

    # -------- bin_accuracy --------
    sub1 = sub.sort_values("bin_accuracy", ascending=False)

    plt.figure(figsize=(10, 4))
    plt.bar(
        sub1["model"].astype(str),
        sub1["bin_accuracy"],
        yerr=sub1["bin_accuracy_std"],
        capsize=4,
    )
    plt.xticks(rotation=45, ha="right")
    plt.ylabel("bin_accuracy")
    plt.title(f"{c} — bin_accuracy by model ({condition})")
    plt.tight_layout()
    plt.show()

    # -------- cont_AP_global --------
    sub2 = sub.sort_values("cont_AP_global", ascending=False)

    plt.figure(figsize=(10, 4))
    plt.bar(
        sub2["model"].astype(str),
        sub2["cont_AP_global"],
        yerr=sub2["cont_AP_global_std"],
        capsize=4,
    )
    plt.xticks(rotation=45, ha="right")
    plt.ylabel("cont_AP_global")
    plt.title(f"{c} — cont_AP_global by model ({condition})")
    plt.tight_layout()
    plt.show()


## Notes

- Version inference:
  - `*_v2.jsonl` → `v2`
  - `*.jsonl` without a `_vN` suffix → treated as `v1`
- If multiple files exist for the same (criterion, condition, version), the notebook uses the **most recently modified** file.
- To evaluate a different manifest, update `MANIFEST_JSONL` and/or tighten `CONDITIONS` patterns.
